# Load GUNDAM engine state from a ROOT output

This notebook loads the engine configuration from a GUNDAM output ROOT file. By default it restores the saved pre-fit data histograms, so `dataType` and `seed` are not needed. It can also load only the stored config and let GUNDAM build data from an explicit `dataType` and `seed`.

In [1]:
nCpuThreads = 3
gundamLibPath = "/Users/nadrino/Documents/Work/Install/gundam/lib"
workDir = "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024"

# GUNDAM output file containing config, pre-fit data histograms, and optionally
# FitterEngine/postFit/Migrad/parameterStateAfterMinimize_TNamed.
outputRootPath = "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/./outputs/fit/gundamFitter_configOA2024_With_v12ProdRun45_noEigen_onlyFlux_Asimov_ToyFit.root"

# Optional. If set, this config is used instead of the config stored in outputRootPath.
# outputRootPath can still be used to load the post-fit state.
configPath = None

overrideList = []

# Mode 1, default: restore the saved data histograms from the ROOT file.
loadDataHistograms = True
loadPostFitState = True

# Mode 2: load only the stored config, then let GUNDAM build the data.
# loadDataHistograms = False
# dataType = "Toy"  # "Asimov", "Toy", or "RealData"
# seed = 12345

In [2]:
import sys
from pathlib import Path

repoRoot = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
srcPath = repoRoot / "src"
if srcPath.exists() and str(srcPath) not in sys.path:
    sys.path.insert(0, str(srcPath))

from gundam_interface import GundamInterface, GundamLoader, GundamRuntime

In [3]:
runtimeOptions = dict(
    loader=GundamLoader(gundamLibPath=gundamLibPath),
    workDir=workDir,
    nCpuThreads=nCpuThreads,
    configPath=configPath,
    outputRootPath=outputRootPath,
    loadDataHistograms=loadDataHistograms,
    loadPostFitState=loadPostFitState,
    overrideList=overrideList,
)
if not loadDataHistograms:
    runtimeOptions.update(dataType=dataType, randomSeed=seed)

runtime = GundamRuntime(**runtimeOptions)

runtime.toDict(includeConfigJsonString=False)

{'nCpuThreads': 3,
 'workDir': '/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024',
 'dataType': 'Asimov',
 'loader': {'moduleName': 'GUNDAM',
  'gundamLibPath': '/Users/nadrino/Documents/Work/Install/gundam/lib'},
 'outputRootPath': '/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/outputs/fit/gundamFitter_configOA2024_With_v12ProdRun45_noEigen_onlyFlux_Asimov_ToyFit.root',
 'loadPostFitState': True,
 'overrideList': []}

In [4]:
gundam = GundamInterface(runtime)
gundam.configure()
gundam.initialize()

2026.07.12 18:04:04  INFO ConfigUtils: Extracting config file for fitter file: /Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/outputs/fit/gundamFitter_configOA2024_With_v12ProdRun45_noEigen_onlyFlux_Asimov_ToyFit.root
2026.07.12 18:04:04  WARN ConfigUtils: /fitterEngineConfig: key "allParamVariations" has been renamed. Use "paramVariationsSigmas" instead.
2026.07.12 18:04:04  WARN ConfigUtils: /fitterEngineConfig: key "scanConfig" has been renamed. Use "parameterScannerConfig" instead.
2026.07.12 18:04:04 ALERT ConfigUtils: /fitterEngineConfig: field "propagatorConfig" should be moved to: fitterEngineConfig/likelihoodInterfaceConfig/propagatorConfig
2026.07.12 18:04:04  WARN ConfigUtils: /fitterEngineConfig/minimizerConfig: key "max_iter" has been renamed. Use "maxIterations" instead.
2026.07.12 18:04:04  WARN ConfigUtils: /fitterEngineConfig/minimizerConfig: key "max_fcn" has been renamed. Use "maxFcnCalls" instead.
2026.07.12 18:04:04 ALERT ConfigUtils: /fitter

In [5]:
print(f"Initialized GUNDAM with {len(gundam.parameters)} active parameters")
print(f"Loaded post-fit state: {runtime.loadPostFitState}")

for index, parameter in enumerate(gundam.parameters):
    print(
        f"{index:3d}: {parameter.name} "
        f"prior={parameter.prior:.8g} "
        f"step={parameter.stepSize:.8g} "
        f"value={parameter.value:.8g}"
    )

Initialized GUNDAM with 100 active parameters
Loaded post-fit state: True
  0: Flux Systematics/#0 prior=1 step=0.05924376 value=0.93486537
  1: Flux Systematics/#1 prior=1 step=0.052534918 value=0.94857857
  2: Flux Systematics/#2 prior=1 step=0.052933735 value=0.9431547
  3: Flux Systematics/#3 prior=1 step=0.051454976 value=0.91155718
  4: Flux Systematics/#4 prior=1 step=0.073819516 value=0.84186369
  5: Flux Systematics/#5 prior=1 step=0.087584986 value=0.82487506
  6: Flux Systematics/#6 prior=1 step=0.069835737 value=0.88442739
  7: Flux Systematics/#7 prior=1 step=0.049799122 value=0.92074631
  8: Flux Systematics/#8 prior=1 step=0.050205095 value=0.90311313
  9: Flux Systematics/#9 prior=1 step=0.070165897 value=0.8760769
 10: Flux Systematics/#10 prior=1 step=0.11426228 value=0.81943851
 11: Flux Systematics/#11 prior=1 step=0.097235633 value=0.92764861
 12: Flux Systematics/#12 prior=1 step=0.065676759 value=0.94075522
 13: Flux Systematics/#13 prior=1 step=0.071617626 value

In [7]:
currentValues = gundam.getParameterValues()
currentLlh = gundam.evaluateLlh()

print(f"Current LLH: {currentLlh:.8g}")
print("Current parameters:")
for name, physicalValue in zip(gundam.parameterNames, currentValues):
    print(f"  - {name}: physical={physicalValue:.8g}")

Current LLH: 4068.9475
Current parameters:
  - Flux Systematics/#0: physical=0.93486537
  - Flux Systematics/#1: physical=0.94857857
  - Flux Systematics/#2: physical=0.9431547
  - Flux Systematics/#3: physical=0.91155718
  - Flux Systematics/#4: physical=0.84186369
  - Flux Systematics/#5: physical=0.82487506
  - Flux Systematics/#6: physical=0.88442739
  - Flux Systematics/#7: physical=0.92074631
  - Flux Systematics/#8: physical=0.90311313
  - Flux Systematics/#9: physical=0.8760769
  - Flux Systematics/#10: physical=0.81943851
  - Flux Systematics/#11: physical=0.92764861
  - Flux Systematics/#12: physical=0.94075522
  - Flux Systematics/#13: physical=0.93220909
  - Flux Systematics/#14: physical=0.98476139
  - Flux Systematics/#15: physical=0.96836057
  - Flux Systematics/#16: physical=0.9287028
  - Flux Systematics/#17: physical=0.92340808
  - Flux Systematics/#18: physical=0.92305327
  - Flux Systematics/#19: physical=0.89697235
  - Flux Systematics/#20: physical=0.91853624
  - 